In [1]:
import sys
sys.path.append("/home/hkerino/Documents/next_oso")
from data_functions import dataset
from Alise import ALISE, utils_ALISE, sse, temporal_proj, transformer
from einops import rearrange, repeat

PATH_TO_DATA_s2 = "/home/hkerino/Documents/nextoso-dataset/data/10m/s2"
PATH_MNT = "/home/hkerino/Documents/nextoso-dataset/data/10m/mnt"
PATH_TO_DATA_label = "/home/hkerino/Documents/nextoso-dataset/data/10m/labels"
PATH_TO_OSO_CLASSES = (
    "/home/hkerino/Documents/nextoso-dataset/data/auxiliary/OSO/oso_classes.csv"
)

In [2]:
batch_size = 1

dl_train, dl_val, dl_test = dataset.get_loaders(
    path_s2=PATH_TO_DATA_s2,
    path_mnt=PATH_MNT,
    path_label=PATH_TO_DATA_label,
    path_oso_class=PATH_TO_OSO_CLASSES,
    folds_train=[1],
    folds_val=[4],
    folds_test=[5],
    batch_size=batch_size,
    num_worker=4,
    norm=False,
    norm_alise=True,
    shuffle=True,
    label_type="max_class",
    use_spatial_coords=False,
    mask_needed=False,
    sequence_packing=False,
    pass_id=False,
)

In [3]:
inputs, label = next(iter(dl_train))
print(inputs[0].shape)
print(inputs[1].shape)

torch.Size([1, 10, 45, 128, 128])
torch.Size([1, 45])


## Spatial encoding

In [4]:
x_in = inputs[0].permute(0, 2, 1, 3, 4).contiguous()
B, T, C, H, W = x_in.shape

#U-NET
unet = sse.Unet(planes=64)
x = rearrange(x_in, "B T C H W -> (B T) C H W ")
out_unet = unet(x)
out_unet =  rearrange(out_unet, "(B T) C H W -> B T C H W", B=B, T=T, H=H, W=W)
# Positional Encoding
pe = sse.PositionalEncoder(d=64)
out_pe = pe(inputs[1])
out_pe = repeat(out_pe, 'B T C -> B T C H W', H=H, W=W)

# Addition
out = out_unet + out_pe
print(out.shape)

#As one bloc
spatial_encoder = sse.PatchEmbedding(sse= unet, tpe=pe)
out_se = spatial_encoder(x_in, inputs[1])
print(out_se.shape)

(out == out_se).all()

torch.Size([1, 45, 64, 128, 128])
torch.Size([1, 45, 64, 128, 128])


tensor(True)

## Temporal encoding

In [13]:
x_in = rearrange(out_se, "B T C H W -> (B H W ) T C")
pad_mask = repeat(inputs[1] == 0, "B T -> (B H W) T", H=H, W=W) #create mask where dates "0" are flagged True

temporal_encoder = transformer.Transformer(d_model=64)
out_temp = temporal_encoder(x_in, src_key_padding_mask=pad_mask)
out_temp.shape

torch.Size([16384, 45, 64])

## Temporal projection

In [21]:
proj_temp = temporal_proj.LearnedQMultiHeadAttention(n_head=4,d_in=64, d_k=32, n_q=5)
out_proj = proj_temp(out_temp, pad_mask=~pad_mask)
print(out_proj.shape)

torch.Size([16384, 5, 64])


## Decoding

In [26]:
mlp = utils_ALISE.MLP_ONNX(mlp=[5*64,3*64,64,25])
out = mlp(out_proj)
print(out.shape)

torch.Size([16384, 25])
